# Kámárí · 04 · Push to Hugging Face  (the single upload step · Colab)

Everything from notebooks 01–03 is uploaded **here, once**:

- manifests (`manifest_{public,train,benchmark}_v0.parquet`)
- licence-permitted aligned crops
- EDA figures + `data_quality_report.md`
- dataset registry, `licenses.md`, generated dataset card

Creates two datasets under `HF_NAMESPACE`: **`dataset-registry-v0`** and **`kamari-safe-open-v0`** (frozen benchmark). Raw crops upload only for redistribution-cleared licences (e.g. FairFace CC-BY-4.0).

In [ ]:
# --- Kámárí bootstrap: works on Google Colab and locally ---
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import userdata
        for _k in ['HF_TOKEN','HF_NAMESPACE','KAMARI_REPO_URL']:
            try: os.environ.setdefault(_k, userdata.get(_k))
            except Exception: pass
    except Exception: pass
    if not pathlib.Path('/content/kamari/data').exists():
        os.system(f"git clone {os.environ.get('KAMARI_REPO_URL','')} /content/kamari")
    os.system('pip install -q -r /content/kamari/requirements-data.txt')
    REPO = pathlib.Path('/content/kamari')
else:
    REPO = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (c/'data'/'registry').exists()), pathlib.Path.cwd().parent)
    from dotenv import load_dotenv; load_dotenv(REPO/'.env')
sys.path.append(str(REPO))
print('REPO =', REPO, '| Colab:', IN_COLAB)

In [ ]:
import pandas as pd
from data.hf_utils import HF, dataset_card
hf = HF()  # reads HF_TOKEN + HF_NAMESPACE
MAN = REPO / 'data' / 'manifests'; CARDS = REPO / 'data' / 'cards'
public = pd.read_parquet(MAN / 'manifest_public_v0.parquet')
print('namespace:', hf.namespace, '| manifest rows:', len(public))

In [ ]:
# 1) dataset-registry-v0: manifests + registry + licences + EDA + card
DS = 'dataset-registry-v0'
for split in ['public', 'train', 'benchmark']:
    hf.push_manifest(pd.read_parquet(MAN / f'manifest_{split}_v0.parquet'), DS, split=split)
for local, repo_path in [
    (REPO/'data'/'registry'/'datasets_free_open.yaml', 'registry/datasets_free_open.yaml'),
    (REPO/'data'/'registry'/'licenses.md',            'registry/licenses.md'),
    (CARDS/'data_quality_report.md',                  'reports/data_quality_report.md')]:
    if local.exists(): hf.upload_file(str(local), DS, repo_path, repo_type='dataset')
for png in (CARDS/'eda').glob('*.png'):
    hf.upload_file(str(png), DS, f'reports/eda/{png.name}', repo_type='dataset')
card = dataset_card(hf.namespace, len(public), sorted(public['dataset'].unique().tolist()))
(CARDS/'DATASET_CARD.md').write_text(card)
hf.upload_file(str(CARDS/'DATASET_CARD.md'), DS, 'README.md', repo_type='dataset')
print('uploaded registry ->', hf.repo_id(DS))

In [ ]:
# 2) Licence-cleared crops -> dataset-registry-v0/crops/
import shutil, tempfile
REDISTRIBUTABLE = {'CC-BY-4.0'}
share = public[public['license'].isin(REDISTRIBUTABLE)]
if len(share):
    tmp = pathlib.Path(tempfile.mkdtemp()) / 'crops'; tmp.mkdir(parents=True)
    for p in share['path']:
        src = REPO / p
        if src.exists(): shutil.copy(src, tmp / src.name)
    hf.upload_folder(str(tmp), DS, repo_type='dataset', path_in_repo='crops')
    print('uploaded', len(list(tmp.glob('*'))), 'crops')
else:
    print('no licence-cleared crops to upload')

In [ ]:
# 3) kamari-safe-open-v0: frozen benchmark manifest + card
BENCH = 'kamari-safe-open-v0'
hf.push_manifest(pd.read_parquet(MAN / 'manifest_benchmark_v0.parquet'), BENCH, split='benchmark')
hf.upload_file(str(CARDS/'DATASET_CARD.md'), BENCH, 'README.md', repo_type='dataset')
print('uploaded benchmark ->', hf.repo_id(BENCH))

In [ ]:
print('Done. Datasets on Hugging Face:')
print('  registry :', f'https://huggingface.co/datasets/{hf.repo_id("dataset-registry-v0")}')
print('  benchmark:', f'https://huggingface.co/datasets/{hf.repo_id("kamari-safe-open-v0")}')
print('\nPaste these links back to kick off CNN + Gemma training on Modal.')